# Batch analysis: C++ vs Python parity

Requires `.read-only/RingDownAnalysis/data` (CSV/MAT fixtures) or set `RINGDOWN_EXAMPLES_DATA`.

This workflow analyzes all top-level CSV/MAT files in the data directory and writes reference-notebook-style plots for both Python and C++ outputs. The notebook cell prints Python and C++ compute times before running the comparison command, which writes plots before returning nonzero when parity mismatches remain.

The C++ step uses the Release binary and enables progress output. By default, the C++ batch report omits raw waveform arrays for speed; add `--notebook-report` to the C++ command when waveform-heavy time-series plots are needed.

```bash
.venv/bin/python examples/python/export_batch_reference.py --n-jobs -1
cmake --build build/release
build/release/examples/batch_analysis_example --workers 2 --progress
.venv/bin/python examples/python/compare_batch_analysis.py \
  --py-report results/examples/batch_analysis_py/batch_report.json \
  --cpp-report results/examples/batch_analysis_cpp/batch_report.json \
  --plot results/examples/batch_analysis_cpp/f_nls_overlay.png \
  --plot-dir results/examples/batch_analysis_plots
```


In [1]:
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "examples/python/export_batch_reference.py").exists():
    ROOT = ROOT.parent

PYTHON = ROOT / ".venv/bin/python"
RELEASE_BUILD = ROOT / "build/release"
CPP_BATCH = RELEASE_BUILD / "examples/batch_analysis_example"

if not RELEASE_BUILD.exists():
    raise FileNotFoundError(
        "Release build directory is missing; configure it with: "
        "cmake -S . -B build/release -DCMAKE_BUILD_TYPE=Release"
    )


def run_timed(label, args):
    start = time.perf_counter()
    try:
        subprocess.check_call(args, cwd=ROOT)
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label} compute time: {elapsed:.3f} s")


run_timed(
    "Python",
    [str(PYTHON), str(ROOT / "examples/python/export_batch_reference.py"), "--n-jobs", "-1"],
)
subprocess.check_call(["cmake", "--build", str(RELEASE_BUILD)], cwd=ROOT)
if not CPP_BATCH.exists():
    raise FileNotFoundError(f"Release C++ batch binary was not produced: {CPP_BATCH}")
run_timed(
    "C++",
    [str(CPP_BATCH), "--workers", "2", "--progress"],
)
comparison = subprocess.run(
    [
        str(PYTHON),
        str(ROOT / "examples/python/compare_batch_analysis.py"),
        "--py-report",
        str(ROOT / "results/examples/batch_analysis_py/batch_report.json"),
        "--cpp-report",
        str(ROOT / "results/examples/batch_analysis_cpp/batch_report.json"),
        "--plot",
        str(ROOT / "results/examples/batch_analysis_cpp/f_nls_overlay.png"),
        "--plot-dir",
        str(ROOT / "results/examples/batch_analysis_plots"),
    ],
    cwd=ROOT,
    check=False,
)
if comparison.returncode != 0:
    print(f"Comparison exited with status {comparison.returncode}; inspect mismatch output above.")
print("Done.")


Wrote Python batch reference under results/examples/batch_analysis_py
Python compute time: 78.715 s
[ 40%] Built target ringdownanalysis
[ 50%] Built target ringdown_cli
[ 70%] Built target ringdown_tests
[ 80%] Built target ringdown_benchmark_smoke
[ 90%] Built target array_analysis_example
[100%] Built target batch_analysis_example


[batch] analyze_file_start 2/14 elapsed_ms=0 path=/Users/mdovale/Work-local/RingDownAnalysis_Cpp/.read-only/RingDownAnalysis/data/MTS_ringdown_050_laser_out_Test2_20250828_234506 1.csv
[batch] analyze_file_start 1/14 elapsed_ms=0 path=/Users/mdovale/Work-local/RingDownAnalysis_Cpp/.read-only/RingDownAnalysis/data/MTS_ringdown_050_laser_out_Test1_20250828_014146_5e-5to1e-5mbar.csv
[batch] analyze_file_failed 1/14 elapsed_ms=0 path=/Users/mdovale/Work-local/RingDownAnalysis_Cpp/.read-only/RingDownAnalysis/data/MTS_ringdown_050_laser_out_Test1_20250828_014146_5e-5to1e-5mbar.csv message=File size (2,620,328,872 bytes) exceeds maximum allowed (1,073,741,824 bytes): /Users/mdovale/Work-local/RingDownAnalysis_Cpp/.read-only/RingDownAnalysis/data/MTS_ringdown_050_laser_out_Test1_20250828_014146_5e-5to1e-5mbar.csv
[batch] analyze_file_start 3/14 elapsed_ms=0 path=/Users/mdovale/Work-local/RingDownAnalysis_Cpp/.read-only/RingDownAnalysis/data/MTS_ringdown_050_laser_out_Test5_20250901_224711_1.5e

Wrote batch analysis artifacts under results/examples/batch_analysis_cpp
Timing: process_files_ms=17556.951874999999 batch_report_json_ms=0.671292 batch_report_write_ms=0.229792 total_ms=17558.060541999999 notebook_report=false
C++ compute time: 17.622 s
Wrote plot /Users/mdovale/Work-local/RingDownAnalysis_Cpp/results/examples/batch_analysis_cpp/f_nls_overlay.png
Wrote 6 Python plot(s) under /Users/mdovale/Work-local/RingDownAnalysis_Cpp/results/examples/batch_analysis_plots/python
Wrote 5 C++ plot(s) under /Users/mdovale/Work-local/RingDownAnalysis_Cpp/results/examples/batch_analysis_plots/cpp
Wrote comparison plot /Users/mdovale/Work-local/RingDownAnalysis_Cpp/results/examples/batch_analysis_plots/comparison_f_nls_overlay.png
OK: batch report fields match within tolerances.
Done.
